# INV-M365-CD — Authoritative Persona Post-H5 Parity Correction v1

**Purpose:** Prove that the current post-H5 parity-correction package cannot truthfully unblock `M1` because department certification remains both stale and out of scope.

**Lemma mapping:** `docs/ma/lemmas/L82_m365_authoritative_persona_post_h5_parity_correction_v1.md`

**Invariant support:** `invariants/lemmas/L82_m365_authoritative_persona_post_h5_parity_correction_v1.yaml`

**Expected results:** Deterministically show the live registry is at `59 total / 54 active / 5 planned`, department certification still claims `34 active / 25 planned`, and the current correction package allowlist cannot satisfy the full `M1` department-certification validation slice.


This notebook is governance prerequisite evidence, not the corrective implementation itself. It exists to prove the scope gap mechanically before any follow-on scope-correction package is created.


In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path, PurePosixPath

import yaml

repo = Path.cwd().resolve()
while not (repo / "registry" / "persona_registry_v2.yaml").exists():
    if repo.parent == repo:
        raise RuntimeError("repo root not found")
    repo = repo.parent

notebook_path = repo / "notebooks" / "m365" / "INV-M365-CD-authoritative-persona-post-h5-parity-correction-v1.ipynb"
parity_plan_path = repo / "plans" / "m365-authoritative-persona-post-h5-parity-correction" / "m365-authoritative-persona-post-h5-parity-correction.yaml"
merge_plan_path = repo / "plans" / "m365-authoritative-persona-humanization-merge-to-development" / "m365-authoritative-persona-humanization-merge-to-development.md"
registry_path = repo / "registry" / "persona_registry_v2.yaml"
department_cert_path = repo / "registry" / "department_certification_v1.yaml"
department_verifier_path = repo / "scripts" / "ci" / "verify_department_certification_v1.py"
department_test_path = repo / "tests" / "test_department_certification_v1.py"
verification_path = repo / "configs" / "generated" / "authoritative_persona_post_h5_parity_correction_v1_verification.json"

assert parity_plan_path.exists(), parity_plan_path
assert merge_plan_path.exists(), merge_plan_path
assert registry_path.exists(), registry_path
assert department_cert_path.exists(), department_cert_path
assert department_verifier_path.exists(), department_verifier_path
assert department_test_path.exists(), department_test_path


Load the current blocker package, the merge package, and the live certification surfaces. The goal is to prove whether the current correction package can legally satisfy the `M1` validation boundary it claims to unblock.


In [ ]:
parity_plan = yaml.safe_load(parity_plan_path.read_text(encoding="utf-8"))
merge_plan_text = merge_plan_path.read_text(encoding="utf-8")
registry = yaml.safe_load(registry_path.read_text(encoding="utf-8"))
department_cert = yaml.safe_load(department_cert_path.read_text(encoding="utf-8"))
department_verifier_text = department_verifier_path.read_text(encoding="utf-8")
department_test_text = department_test_path.read_text(encoding="utf-8")

allowlist = parity_plan["scope"]["file_allowlist"]
department_scope_targets = [
    "registry/department_certification_v1.yaml",
    "docs/commercialization/m365-department-certification-v1.md",
    "scripts/ci/verify_department_certification_v1.py",
    "tests/test_department_certification_v1.py",
]

def covered(path: str) -> bool:
    pure = PurePosixPath(path)
    return any(pure.match(pattern) for pattern in allowlist)

scope_coverage = {path: covered(path) for path in department_scope_targets}
missing_scope_targets = sorted(path for path, is_covered in scope_coverage.items() if not is_covered)

registry_summary = registry["summary"]
department_kpis = department_cert["kpis"]
department_truth_mismatch = {
    "active_personas": {
        "registry": registry_summary["active_personas"],
        "department_certification": department_kpis["active_department_personas"],
    },
    "planned_personas": {
        "registry": registry_summary["planned_personas"],
        "department_certification": department_kpis["planned_department_personas"],
    },
    "registry_backed_personas": {
        "registry": registry_summary["registry_backed_personas"],
        "department_certification": department_kpis["registry_backed_department_personas"],
    },
    "contract_only_personas": {
        "registry": registry_summary["persona_contract_only_personas"],
        "department_certification": department_kpis["contract_only_department_personas"],
    },
}
stale_department_truth = any(values["registry"] != values["department_certification"] for values in department_truth_mismatch.values())

m1_requires_department_validation = (
    "verify_department_certification_v1.py" in merge_plan_text
    and "test_department_certification_v1.py" in merge_plan_text
)
department_validation_binds_to_registry = (
    "persona_registry_v2.yaml" in department_verifier_text
    and "active_department_personas mismatch" in department_verifier_text
    and "planned_department_personas mismatch" in department_verifier_text
    and "registry_backed_department_personas mismatch" in department_verifier_text
    and "contract_only_department_personas mismatch" in department_verifier_text
    and "active_department_personas" in department_test_text
    and "planned_department_personas" in department_test_text
)

scope_correction_required = bool(
    m1_requires_department_validation
    and department_validation_binds_to_registry
    and stale_department_truth
    and missing_scope_targets
)

verification = {
    "status": "passed",
    "lemma_id": "L82",
    "notebook_backing": {
        "path": str(notebook_path.relative_to(repo)),
        "role": "governance_prerequisite_evidence",
    },
    "current_package": {
        "plan_id": parity_plan["plan_id"],
        "allowlist_entry_count": len(allowlist),
    },
    "merge_package": {
        "plan_id": "plan:m365-authoritative-persona-humanization-merge-to-development",
        "requires_department_validation": m1_requires_department_validation,
    },
    "registry_truth": {
        "total_personas": registry_summary["total_personas"],
        "active_personas": registry_summary["active_personas"],
        "planned_personas": registry_summary["planned_personas"],
        "registry_backed_personas": registry_summary["registry_backed_personas"],
        "contract_only_personas": registry_summary["persona_contract_only_personas"],
    },
    "department_certification_truth": {
        "total_personas": department_kpis["total_department_personas"],
        "active_personas": department_kpis["active_department_personas"],
        "planned_personas": department_kpis["planned_department_personas"],
        "registry_backed_personas": department_kpis["registry_backed_department_personas"],
        "contract_only_personas": department_kpis["contract_only_department_personas"],
    },
    "department_validation_binds_to_registry": department_validation_binds_to_registry,
    "scope_coverage": scope_coverage,
    "missing_scope_targets": missing_scope_targets,
    "department_truth_mismatch": department_truth_mismatch,
    "scope_correction_required": scope_correction_required,
    "decision": "new scope-correction package required before M1 replay" if scope_correction_required else "current package sufficient",
}

hash_payload = json.dumps(verification, sort_keys=True, separators=(",", ":")).encode("utf-8")
verification["deterministic_hash"] = hashlib.sha256(hash_payload).hexdigest()

verification_path.parent.mkdir(parents=True, exist_ok=True)
verification_path.write_text(json.dumps(verification, indent=2, sort_keys=True) + "\n", encoding="utf-8")
verification


The notebook must fail closed unless it proves all four ingredients of the scope gap: `M1` requires department validation, the current package does not cover those files, department certification is stale against the live registry, and the verifier/test pair mechanically binds that stale surface back to the registry.


In [ ]:
assert verification["registry_truth"] == {
    "total_personas": 59,
    "active_personas": 54,
    "planned_personas": 5,
    "registry_backed_personas": 54,
    "contract_only_personas": 5,
}
assert verification["department_certification_truth"] == {
    "total_personas": 59,
    "active_personas": 34,
    "planned_personas": 25,
    "registry_backed_personas": 34,
    "contract_only_personas": 25,
}
assert verification["merge_package"]["requires_department_validation"] is True
assert verification["department_validation_binds_to_registry"] is True
assert verification["missing_scope_targets"] == [
    "docs/commercialization/m365-department-certification-v1.md",
    "registry/department_certification_v1.yaml",
    "scripts/ci/verify_department_certification_v1.py",
    "tests/test_department_certification_v1.py",
]
assert verification["scope_correction_required"] is True
print("post-H5 parity scope gap proven")
